In [65]:
import cv2
import mediapipe as mp
import numpy as np
import math
import random
import time
import glfw
import json
import os

from OpenGL.GL import *
from OpenGL.GLU import *

In [66]:
WIDTH, HEIGHT = 1700, 750         
DASHBOARD_WIDTH = 500             
CUBE_VIEW_WIDTH = WIDTH - DASHBOARD_WIDTH 

YAW = -30.0
PITCH = 25.0
CAMERA_DISTANCE = 7.0

In [67]:
COLOR_BG = (18, 22, 28)
COLOR_PANEL = (28, 34, 44)
COLOR_BORDER = (60, 72, 90)
COLOR_ACCENT = (0, 215, 255)
COLOR_GREEN = (0, 230, 120)
COLOR_RED = (80, 80, 255)
COLOR_TEXT = (240, 240, 240)
COLOR_MUTED = (160, 175, 195)
COLOR_BUTTON = (45, 55, 72)
COLOR_HOVER = (70, 95, 130)

In [68]:
COLORS_3D = {
    'W': (0.95, 0.95, 0.95),  # White
    'Y': (1.00, 0.85, 0.00),  # Yellow
    'G': (0.00, 0.75, 0.10),  # Green
    'B': (0.00, 0.30, 0.85),  # Blue
    'R': (0.85, 0.05, 0.05),  # Red
    'O': (1.00, 0.40, 0.00)   # Orange
}

In [69]:
LEADERBOARD_FILE = "rubiks_leaderboard.json"

def load_leaderboard():
    if os.path.exists(LEADERBOARD_FILE):
        try:
            with open(LEADERBOARD_FILE, 'r') as f:
                return json.load(f)
        except Exception:
            pass
    return []

In [70]:
def save_score(score, elapsed, moves, difficulty):
    board = load_leaderboard()
    entry = {
        "score": score,
        "time": round(elapsed, 2),
        "moves": moves,
        "difficulty": difficulty,
        "date": time.strftime("%Y-%m-%d %H:%M")
    }
    board.append(entry)
    board = sorted(board, key=lambda x: x['score'], reverse=True)[:5]
    with open(LEADERBOARD_FILE, 'w') as f:
        json.dump(board, f, indent=2)
    return board

In [71]:
def create_solved_cube():
    cube = {}
    for x in [-1, 0, 1]:
        for y in [-1, 0, 1]:
            for z in [-1, 0, 1]:
                stickers = {}
                if z == 1:  stickers[(0, 0, 1)] = 'G'
                if z == -1: stickers[(0, 0, -1)] = 'B'
                if x == 1:  stickers[(1, 0, 0)] = 'R'
                if x == -1: stickers[(-1, 0, 0)] = 'O'
                if y == 1:  stickers[(0, 1, 0)] = 'W'
                if y == -1: stickers[(0, -1, 0)] = 'Y'
                cube[(x, y, z)] = stickers
    return cube

cube = create_solved_cube()

In [72]:
animating = False
anim_axis = np.array([0, 0, 1])
anim_layer = 1
anim_angle = 0.0
anim_target_angle = 0.0
anim_speed = 15.0

In [73]:
game_mode = "CHALLENGE"
difficulty = "MEDIUM"
show_leaderboard = False
timer_running = False
start_time = 0.0
elapsed_time = 0.0
move_counter = 0
current_score = 0
last_completed_score = 0
completion_modal = False
game_over_modal = False
inverse_history = []
solve_queue = []
last_solve_step_time = 0.0
SOLVE_STEP_DELAY = 1.2

In [74]:
scanning_mode = False
scan_step_index = 0
last_scan_time = 0.0
face_names = ["Front", "Right", "Back", "Left", "Top", "Bottom"]
scanned_face_grids = {face: None for face in face_names}
all_faces_scanned = False
step_instructions = []


In [75]:
hover_button_id = None
hover_start_time = 0.0
HOVER_DURATION = 1.0
scramble_gesture_start = 0.0
active_alert_message = ""
alert_expire_time = 0.0

In [76]:
def trigger_popup_alert(msg, duration=2.5):
    global active_alert_message, alert_expire_time
    active_alert_message = msg
    alert_expire_time = time.time() + duration

In [77]:
def rotate_vector(vector, axis, quarter):
    v = np.array(vector, dtype=float)
    a = np.array(axis, dtype=float) / np.linalg.norm(axis)
    angle = quarter * (math.pi / 2)
    cos_a, sin_a = math.cos(angle), math.sin(angle)
    rot = v * cos_a + np.cross(a, v) * sin_a + a * np.dot(a, v) * (1 - cos_a)
    return tuple(np.round(rot).astype(int))

In [78]:
def commit_layer_rotation(axis, layer, quarter):
    global cube
    new_cube = {}
    a = np.array(axis)
    for pos, stickers in cube.items():
        if np.dot(pos, a) == layer:
            new_pos = rotate_vector(pos, a, quarter)
            new_stickers = {rotate_vector(n, a, quarter): c for n, c in stickers.items()}
            new_cube[new_pos] = new_stickers
        else:
            new_cube[pos] = stickers.copy()
    cube = new_cube

In [79]:
def start_layer_animation(axis, layer, quarter):
    global animating, anim_axis, anim_layer, anim_angle, anim_target_angle
    if animating:
        return
    animating = True
    anim_axis = np.array(axis)
    anim_layer = layer
    anim_angle = 0.0
    anim_target_angle = quarter * 90.0

In [80]:
def update_animation():
    global animating, anim_angle, anim_target_angle
    if not animating:
        return

    step = math.copysign(anim_speed, anim_target_angle - anim_angle)
    if abs(anim_target_angle - anim_angle) <= abs(step):
        quarter = 1 if anim_target_angle > 0 else -1
        commit_layer_rotation(anim_axis, anim_layer, quarter)
        animating = False
        anim_angle = 0.0
        check_solved_state()
    else:
        anim_angle += step

In [81]:
def record_move(axis, layer, quarter):
    global move_counter, inverse_history, timer_running, start_time, completion_modal, game_over_modal
    if game_over_modal:
        return
    completion_modal = False
    move_counter += 1
    inverse_history.append((axis, layer, -quarter))
    if game_mode == "CHALLENGE" and not timer_running and not is_cube_solved():
        timer_running = True
        start_time = time.time()

In [82]:
def trigger_scramble():
    global cube, move_counter, inverse_history, solve_queue, timer_running, elapsed_time, completion_modal, game_over_modal, scanned_face_grids, all_faces_scanned, step_instructions
    cube = create_solved_cube()
    move_counter = 0
    inverse_history.clear()
    solve_queue.clear()
    timer_running = False
    elapsed_time = 0.0
    completion_modal = False
    game_over_modal = False
    all_faces_scanned = False
    scanned_face_grids = {face: None for face in face_names}
    step_instructions.clear()
    
    length_map = {"EASY": 4, "MEDIUM": 10, "HARD": 18}
    num_moves = length_map.get(difficulty, 10)
    
    possible_moves = [((1, 0, 0), 1), ((1, 0, 0), -1),
                      ((0, 1, 0), 1), ((0, 1, 0), -1),
                      ((0, 0, 1), 1), ((0, 0, 1), -1)]
    for _ in range(num_moves):
        axis, layer = random.choice(possible_moves)
        quarter = random.choice([-1, 1])
        commit_layer_rotation(axis, layer, quarter)
        inverse_history.append((axis, layer, -quarter))

In [83]:
def prepare_solver():
    global solve_queue, inverse_history, step_instructions, last_solve_step_time
    if game_mode == "CHALLENGE":
        trigger_popup_alert("Challenge Mode: Auto-solver is disabled!")
        return
        
    if not all_faces_scanned:
        trigger_popup_alert("Please scan all 6 cube faces first!")
        return
        
    if not solve_queue and inverse_history:
        solve_queue = list(reversed(inverse_history))
        last_solve_step_time = time.time()
        
        step_instructions.clear()
        
        # Correctly map axis and layer to standard Rubik's notation (R, L, U, D, F, B)
        for ax, layer, q in solve_queue:
            ax_tuple = tuple(ax)
            if ax_tuple == (1, 0, 0):
                face_char = "R" if layer == 1 else ("L" if layer == -1 else "M")
            elif ax_tuple == (0, 1, 0):
                face_char = "U" if layer == 1 else ("D" if layer == -1 else "E")
            elif ax_tuple == (0, 0, 1):
                face_char = "F" if layer == 1 else ("B" if layer == -1 else "S")
            else:
                face_char = "M"

            modifier = "'" if q < 0 else ""
            if q in [-2, 2]: modifier = "2"
            step_instructions.append(f"{face_char}{modifier}")
            
        inverse_history.clear()
        trigger_popup_alert("Practice Solver Ready!", 2.5)
    elif not solve_queue:
        trigger_popup_alert("Cube is already solved!", 2.0)

In [84]:
def process_solver_queue():
    global solve_queue, last_solve_step_time
    if solve_queue and not animating and (time.time() - last_solve_step_time > SOLVE_STEP_DELAY):
        axis, layer, quarter = solve_queue.pop(0)
        start_layer_animation(axis, layer, quarter)
        last_solve_step_time = time.time()
        if step_instructions:
            step_instructions.pop(0)

In [85]:
def is_cube_solved():
    return cube == create_solved_cube()

In [86]:
def check_solved_state():
    global timer_running, elapsed_time, current_score, last_completed_score, completion_modal, game_over_modal
    if is_cube_solved() and timer_running:
        timer_running = False
        elapsed_time = time.time() - start_time
        
        base_score = 500
        time_penalty = int(elapsed_time * 2)
        move_penalty = move_counter * 5
        diff_multiplier = {"EASY": 1.0, "MEDIUM": 1.5, "HARD": 2.5}.get(difficulty, 1.0)
        
        calculated_score = max(50, int((base_score - time_penalty - move_penalty) * diff_multiplier))
        if game_mode == "PRACTICE":
            calculated_score = 50
            
        current_score += calculated_score
        last_completed_score = calculated_score
        completion_modal = True
        save_score(current_score, elapsed_time, move_counter, difficulty)

In [87]:
def check_hard_mode_timeout():
    global timer_running, game_over_modal
    if timer_running and difficulty == "HARD" and game_mode == "CHALLENGE":
        cur_elapsed = time.time() - start_time
        if cur_elapsed > 60.0:
            timer_running = False
            game_over_modal = True
            trigger_popup_alert("TIME UP! Hard Mode Failed (60s Expired)", 4.0)

In [88]:
def setup_opengl():
    glEnable(GL_DEPTH_TEST)
    glEnable(GL_LIGHTING)
    glEnable(GL_LIGHT0)
    glEnable(GL_COLOR_MATERIAL)
    glColorMaterial(GL_FRONT_AND_BACK, GL_AMBIENT_AND_DIFFUSE)
    glLightfv(GL_LIGHT0, GL_POSITION, [5.0, 8.0, 10.0, 1.0])
    glLightfv(GL_LIGHT0, GL_AMBIENT, [0.35, 0.35, 0.35, 1.0])
    glLightfv(GL_LIGHT0, GL_DIFFUSE, [0.8, 0.8, 0.8, 1.0])
    glLightfv(GL_LIGHT0, GL_SPECULAR, [1.0, 1.0, 1.0, 1.0])
    glEnable(GL_NORMALIZE)
    glShadeModel(GL_SMOOTH)

In [89]:
def draw_beveled_face():
    h = 0.47
    glMaterialfv(GL_FRONT, GL_SPECULAR, [0.2, 0.2, 0.2, 1.0])
    glMaterialf(GL_FRONT, GL_SHININESS, 10.0)
    glColor3f(0.08, 0.08, 0.08)
    glBegin(GL_QUADS)
    glNormal3f(0, 0, 1)
    glVertex3f(-h, -h, 0.48); glVertex3f(h, -h, 0.48)
    glVertex3f(h, h, 0.48);   glVertex3f(-h, h, 0.48)
    glEnd()

In [90]:
def render_scene():
    update_animation()
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glLoadIdentity()
    gluLookAt(-2.2, 0, CAMERA_DISTANCE, -2.2, 0, 0, 0, 1, 0)
    glRotatef(PITCH, 1, 0, 0)
    glRotatef(YAW, 0, 1, 0)

    for pos, stickers in cube.items():
        glPushMatrix()
        if animating and int(np.dot(pos, anim_axis)) == anim_layer:
            glRotatef(anim_angle, anim_axis[0], anim_axis[1], anim_axis[2])
        glTranslatef(pos[0], pos[1], pos[2])

        for (nx, ny, nz), color_code in stickers.items():
            glPushMatrix()
            if (nx, ny, nz) == (0, 0, 1): pass
            elif (nx, ny, nz) == (0, 0, -1): glRotatef(180, 0, 1, 0)
            elif (nx, ny, nz) == (1, 0, 0):  glRotatef(90, 0, 1, 0)
            elif (nx, ny, nz) == (-1, 0, 0): glRotatef(-90, 0, 1, 0)
            elif (nx, ny, nz) == (0, 1, 0):  glRotatef(-90, 1, 0, 0)
            elif (nx, ny, nz) == (0, -1, 0): glRotatef(90, 1, 0, 0)

            draw_beveled_face()

            glColor3f(*COLORS_3D[color_code])
            glBegin(GL_QUADS)
            glNormal3f(0, 0, 1)
            glVertex3f(-0.41, -0.41, 0.501); glVertex3f(0.41, -0.41, 0.501)
            glVertex3f(0.41, 0.41, 0.501);   glVertex3f(-0.41, 0.41, 0.501)
            glEnd()
            glPopMatrix()
        glPopMatrix()


In [91]:
# Dashboard

def draw_dashboard_overlay(frame, current_hover_id, hover_progress):
    overlay = frame.copy()
    start_x = WIDTH - DASHBOARD_WIDTH
    
    cv2.rectangle(overlay, (start_x, 0), (WIDTH, HEIGHT), COLOR_PANEL, -1)
    cv2.addWeighted(overlay, 0.88, frame, 0.12, 0, frame)
    cv2.line(frame, (start_x, 0), (start_x, HEIGHT), COLOR_BORDER, 2)

    def draw_button(btn_id, x1, y1, x2, y2, label, base_color, text_color):
        is_hovered = (current_hover_id == btn_id)
        bg = COLOR_HOVER if is_hovered else COLOR_BUTTON
        cv2.rectangle(frame, (x1, y1), (x2, y2), bg, -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), base_color, 2 if is_hovered else 1)
        
        if is_hovered and hover_progress > 0:
            fill_w = int((x2 - x1) * hover_progress)
            cv2.rectangle(frame, (x1, y1), (x1 + fill_w, y2), base_color, 2)
            
        cv2.putText(frame, label, (x1 + 12, y1 + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.42, text_color, 2)

    # 1. Header & Controls
    cv2.rectangle(frame, (start_x + 20, 15), (WIDTH - 20, 60), COLOR_PANEL, -1)
    cv2.rectangle(frame, (start_x + 20, 15), (WIDTH - 20, 60), COLOR_BORDER, 1)
    cv2.putText(frame, "AI RUBIK'S CUBE HUD", (start_x + 25, 43), cv2.FONT_HERSHEY_SIMPLEX, 0.48, COLOR_ACCENT, 2)
    
    draw_button("MODE", start_x + 20, 70, start_x + 235, 110, f"MODE: {game_mode}", COLOR_ACCENT, COLOR_TEXT)
    
    diff_color = COLOR_GREEN if difficulty == "EASY" else (COLOR_ACCENT if difficulty == "MEDIUM" else COLOR_RED)
    draw_button("DIFF", start_x + 245, 70, WIDTH - 20, 110, f"DIFF: {difficulty}", diff_color, diff_color)
    
    scan_label = "SCAN 6 FACES" if game_mode == "PRACTICE" else "SCAN (N/A)"
    scan_color = COLOR_ACCENT if game_mode == "PRACTICE" else COLOR_MUTED
    draw_button("SCAN", start_x + 20, 120, start_x + 235, 160, scan_label, scan_color, COLOR_TEXT)
    
    solve_btn_label = "PRACTICE SOLVER" if game_mode == "PRACTICE" else "SOLVE (LOCKED)"
    solve_color = COLOR_GREEN if game_mode == "PRACTICE" else COLOR_MUTED
    draw_button("SOLVE", start_x + 245, 120, WIDTH - 20, 160, solve_btn_label, solve_color, COLOR_TEXT)

    lb_label = "LEADERBOARD: ON" if show_leaderboard else "LEADERBOARD: OFF"
    draw_button("LB_TOGGLE", start_x + 20, 170, WIDTH - 20, 210, lb_label, COLOR_ACCENT, COLOR_TEXT)

    # 2. Middle Panel: Leaderboard OR 6-Face Grid Recognition
    cv2.rectangle(frame, (start_x + 20, 220), (WIDTH - 20, 480), COLOR_PANEL, -1)
    cv2.rectangle(frame, (start_x + 20, 220), (WIDTH - 20, 480), COLOR_BORDER, 1)

    if show_leaderboard:
        cv2.putText(frame, "TOP 5 LEADERBOARD SCORES", (start_x + 30, 248), cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLOR_ACCENT, 2)
        board = load_leaderboard()
        if board:
            for idx, entry in enumerate(board):
                ly = 280 + idx * 36
                txt = f"{idx+1}. Score: {entry['score']} | {entry['difficulty']} | {entry['time']}s"
                cv2.putText(frame, txt, (start_x + 30, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.4, COLOR_TEXT, 1)
                cv2.putText(frame, f"   Moves: {entry['moves']} | {entry['date']}", (start_x + 30, ly + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.33, COLOR_MUTED, 1)
        else:
            cv2.putText(frame, "No high scores recorded yet!", (start_x + 40, 320), cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLOR_MUTED, 1)
    else:
        cv2.putText(frame, "CUBE FACES RECOGNITION (3x3)", (start_x + 30, 245), cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLOR_ACCENT, 2)

        grid_x, grid_y = start_x + 30, 260
        thumb_w, thumb_h = 120, 65
        spacing_x, spacing_y = 15, 8

        for i, face in enumerate(face_names):
            row = i // 3
            col = i % 3
            bx = grid_x + col * (thumb_w + spacing_x)
            by = grid_y + row * (thumb_h + spacing_y + 16)

            cv2.rectangle(frame, (bx, by), (bx + thumb_w, by + thumb_h), COLOR_BUTTON, -1)
            cv2.rectangle(frame, (bx, by), (bx + thumb_w, by + thumb_h), COLOR_BORDER, 1)
            cv2.putText(frame, face, (bx + 8, by + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.35, COLOR_TEXT, 1)

            face_grid = scanned_face_grids[face]
            if face_grid is not None:
                cell_w = (thumb_w - 10) // 3
                cell_h = (thumb_h - 24) // 3
                for r in range(3):
                    for c in range(3):
                        bgr_color = face_grid[r][c]
                        rx = bx + 5 + c * cell_w
                        ry = by + 18 + r * cell_h
                        cv2.rectangle(frame, (rx, ry), (rx + cell_w - 1, ry + cell_h - 1), bgr_color, -1)
            else:
                cv2.putText(frame, "Pending", (bx + 35, by + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.35, COLOR_MUTED, 1)

        if scanning_mode and scan_step_index < len(face_names):
            cv2.putText(frame, f">> Scan Target: {face_names[scan_step_index]} Face <<", (start_x + 30, 465),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, COLOR_GREEN, 2)

    # 3. Telemetry & All-Steps Visible Solver Guide Panel
    cv2.rectangle(frame, (start_x + 20, 490), (WIDTH - 20, 720), COLOR_PANEL, -1)
    cv2.rectangle(frame, (start_x + 20, 490), (WIDTH - 20, 720), COLOR_BORDER, 1)
    
    if game_mode == "CHALLENGE":
        cv2.putText(frame, "CHALLENGE METRICS:", (start_x + 30, 520), cv2.FONT_HERSHEY_SIMPLEX, 0.48, COLOR_RED, 2)
        if difficulty == "HARD":
            rem = 60.0 - (time.time() - start_time) if timer_running else 60.0
            rem = max(0.0, rem)
            time_str = f"TIME REMAINING: {int(rem)}s"
        else:
            cur = (time.time() - start_time) if timer_running else elapsed_time
            time_str = f"ELAPSED TIME: {int(cur)//60:02d}:{int(cur)%60:02d}"

        cv2.putText(frame, time_str, (start_x + 30, 560), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_TEXT, 2)
        cv2.putText(frame, f"MOVE COUNT: {move_counter}", (start_x + 30, 595), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_TEXT, 2)
        cv2.putText(frame, f"TOTAL SCORE: {current_score}", (start_x + 30, 630), cv2.FONT_HERSHEY_SIMPLEX, 0.55, COLOR_ACCENT, 2)
    else:
        cv2.putText(frame, "PRACTICE SOLVER GUIDE (All Steps Visible):", (start_x + 30, 518), cv2.FONT_HERSHEY_SIMPLEX, 0.45, COLOR_ACCENT, 2)
        if step_instructions:
            max_per_col = 9
            col_w = (DASHBOARD_WIDTH - 60) // 2
            for idx, step_text in enumerate(step_instructions):
                col = idx // max_per_col
                row = idx % max_per_col
                if col >= 2: break
                cx = start_x + 30 + col * col_w
                cy = 550 + row * 19
                prefix = f"{idx+1}: "
                color = COLOR_GREEN if idx == 0 else COLOR_TEXT
                cv2.putText(frame, prefix + step_text, (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.38, color, 1)
        else:
            cv2.putText(frame, "Scan 6 faces & click PRACTICE SOLVER.", (start_x + 30, 555), cv2.FONT_HERSHEY_SIMPLEX, 0.4, COLOR_MUTED, 1)

    # Modals for Success / Failure
    if completion_modal:
        cv2.rectangle(frame, (start_x + 30, 320), (WIDTH - 30, 440), (0, 150, 0), -1)
        cv2.putText(frame, "PUZZLE SOLVED!", (start_x + 60, 365), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"+{last_completed_score} Points Awarded", (start_x + 65, 400), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    elif game_over_modal:
        cv2.rectangle(frame, (start_x + 30, 320), (WIDTH - 30, 440), (150, 0, 0), -1)
        cv2.putText(frame, "TIME'S UP! FAILED", (start_x + 50, 365), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, "Hard Mode 60s Expired", (start_x + 70, 400), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)


In [92]:
def get_button_under_cursor(dash_x, y):
    start_x = WIDTH - DASHBOARD_WIDTH
    local_x = dash_x - start_x
    
    if 20 <= local_x <= DASHBOARD_WIDTH - 20:
        if 70 <= y <= 110:
            return "MODE" if local_x <= 215 else "DIFF"
        if 120 <= y <= 160:
            return "SCAN" if local_x <= 215 else "SOLVE"
        if 170 <= y <= 210:
            return "LB_TOGGLE"
    return None

In [93]:
def trigger_button_action(btn_id):
    global game_mode, difficulty, scanning_mode, scan_step_index, show_leaderboard
    if btn_id == "MODE":
        game_mode = "PRACTICE" if game_mode == "CHALLENGE" else "CHALLENGE"
        trigger_popup_alert(f"Mode set to {game_mode}", 1.5)
    elif btn_id == "DIFF":
        diffs = ["EASY", "MEDIUM", "HARD"]
        difficulty = diffs[(diffs.index(difficulty) + 1) % 3]
        trigger_popup_alert(f"Difficulty set to {difficulty}", 1.5)
    elif btn_id == "SCAN":
        if game_mode == "CHALLENGE":
            trigger_popup_alert("Scanning disabled in Challenge Mode", 1.5)
        else:
            scanning_mode = True
            scan_step_index = 0
            trigger_popup_alert("Rotate cube to show face & Pinch to Snap Grid", 3.0)
    elif btn_id == "SOLVE":
        prepare_solver()
    elif btn_id == "LB_TOGGLE":
        show_leaderboard = not show_leaderboard
        trigger_popup_alert(f"Leaderboard View: {'ON' if show_leaderboard else 'OFF'}", 1.2)

In [94]:
def extract_cube_face_grid(frame, box_x, box_y, box_size):
    grid = []
    cell_size = box_size // 3
    for r in range(3):
        row_colors = []
        for c in range(3):
            cx = box_x + c * cell_size + cell_size // 2
            cy = box_y + r * cell_size + cell_size // 2
            b, g, r_val = frame[cy, cx]
            row_colors.append((int(b), int(g), int(r_val)))
        grid.append(row_colors)
    return grid

In [95]:
def main():
    global YAW, PITCH, game_mode, difficulty, elapsed_time, completion_modal, game_over_modal, scanning_mode, scan_step_index, last_scan_time, scanned_face_grids, all_faces_scanned, hover_button_id, hover_start_time, scramble_gesture_start

    if not glfw.init():
        return

    glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
    window = glfw.create_window(CUBE_VIEW_WIDTH, HEIGHT, "OpenGL Context", None, None)
    glfw.make_context_current(window)

    glViewport(0, 0, CUBE_VIEW_WIDTH, HEIGHT)
    glMatrixMode(GL_PROJECTION)
    glLoadIdentity()
    gluPerspective(45.0, CUBE_VIEW_WIDTH / float(HEIGHT), 0.1, 100.0)
    glMatrixMode(GL_MODELVIEW)
    setup_opengl()

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

    window_name = "3D AR Rubik's Cube & Hand Scanner"
    cv2.namedWindow(window_name)

    mp_hands = mp.solutions.hands
    mp_draw = mp.solutions.drawing_utils
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.4, min_tracking_confidence=0.4)

    is_pinching = False
    pinch_start_x, pinch_start_y = 0, 0
    drag_move_triggered = False
    gesture_status = "Ready"
    last_drag_time = 0.0

    dash_start_x = WIDTH - DASHBOARD_WIDTH

    while not glfw.window_should_close(window):
        check_hard_mode_timeout()
        process_solver_queue()
        
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        if frame.shape[1] != WIDTH or frame.shape[0] != HEIGHT:
            frame = cv2.resize(frame, (WIDTH, HEIGHT))

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)

        current_hover_id = None
        hover_progress = 0.0
        cur_time = time.time()

        if results.multi_hand_landmarks and not game_over_modal:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_draw.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_draw.DrawingSpec(color=(0, 255, 255), thickness=2, circle_radius=3),
                    mp_draw.DrawingSpec(color=(255, 0, 255), thickness=2)
                )

            lm = results.multi_hand_landmarks[0].landmark
            ix, iy = int(lm[8].x * WIDTH), int(lm[8].y * HEIGHT)
            tx, ty = int(lm[4].x * WIDTH), int(lm[4].y * HEIGHT)
            pinch_dist = math.sqrt((ix - tx) ** 2 + (iy - ty) ** 2)

            fingers = sum([
                lm[8].y < lm[6].y,
                lm[12].y < lm[10].y,
                lm[16].y < lm[14].y,
                lm[20].y < lm[18].y
            ])

            if fingers == 2 and ix < dash_start_x:
                if scramble_gesture_start == 0.0:
                    scramble_gesture_start = cur_time
                elif cur_time - scramble_gesture_start > 1.5:
                    trigger_scramble()
                    scramble_gesture_start = 0.0
            else:
                scramble_gesture_start = 0.0

            # dashboard
            if ix >= dash_start_x:
                cv2.circle(frame, (ix, iy), 6, (0, 255, 255), -1)
                detected_btn = get_button_under_cursor(ix, iy)
                
                if detected_btn:
                    current_hover_id = detected_btn
                    if hover_button_id != detected_btn:
                        hover_button_id = detected_btn
                        hover_start_time = cur_time
                    
                    elapsed_hover = cur_time - hover_start_time
                    hover_progress = min(1.0, elapsed_hover / HOVER_DURATION)
                    gesture_status = f"Hovering {detected_btn} ({int(hover_progress*100)}%)"

                    if elapsed_hover >= HOVER_DURATION:
                        trigger_button_action(detected_btn)
                        hover_button_id = None
                        hover_start_time = cur_time
                else:
                    hover_button_id = None

            # cube interaction
            else:
                hover_button_id = None
                drag_threshold = {"EASY": 50, "MEDIUM": 75, "HARD": 100}.get(difficulty, 75)

                if pinch_dist < 45:
                    cv2.circle(frame, (ix, iy), 6, (0, 230, 120), -1)
                    if not is_pinching:
                        is_pinching = True
                        pinch_start_x, pinch_start_y = ix, iy
                        drag_move_triggered = False
                        
                        # CAPTURE 3x3 CUBE FACE GRID COLORS
                        if scanning_mode and scan_step_index < len(face_names) and game_mode == "PRACTICE":
                            if cur_time - last_scan_time > 1.5:
                                cur_face = face_names[scan_step_index]
                                
                                box_size = 180
                                start_cx = CUBE_VIEW_WIDTH // 2 - box_size // 2
                                start_cy = HEIGHT // 2 - box_size // 2
                                
                                face_grid = extract_cube_face_grid(frame, start_cx, start_cy, box_size)
                                scanned_face_grids[cur_face] = face_grid
                                trigger_popup_alert(f"Captured {cur_face} Face Colors!", 2.0)
                                
                                scan_step_index += 1
                                last_scan_time = cur_time  # Reset the cooldown timer
                                
                                if scan_step_index >= len(face_names):
                                    scanning_mode = False
                                    all_faces_scanned = True
                                    trigger_popup_alert("All 6 Faces Recognized!", 3.0)
                                else:
                                    trigger_popup_alert(f"Next: Rotate & Show {face_names[scan_step_index]} Face", 2.0)

                    gesture_status = "Pinched (Hold & Swipe)"

                    dx = ix - pinch_start_x
                    dy = iy - pinch_start_y

                    if (math.sqrt(dx * dx + dy * dy) > drag_threshold and 
                        not drag_move_triggered and 
                        not animating and 
                        not scanning_mode and 
                        (cur_time - last_drag_time > 0.8)):
                        
                        if abs(dx) > abs(dy):
                            axis = (0, 1, 0)
                            zone_h = HEIGHT * 0.38
                            if pinch_start_y < zone_h:
                                layer = 1
                            elif pinch_start_y < HEIGHT - zone_h:
                                layer = 0
                            else:
                                layer = -1
                                
                            quarter = 1 if dx > 0 else -1
                            gesture_status = f"Swipe {'Right' if dx > 0 else 'Left'} (Layer {layer})"
                        else:
                            axis = (1, 0, 0)
                            # Custom adjusted zones for width
                            zone_w = CUBE_VIEW_WIDTH * 0.38
                            if pinch_start_x < zone_w:
                                layer = 1
                            elif pinch_start_x < CUBE_VIEW_WIDTH - zone_w:
                                layer = 0
                            else:
                                layer = -1
                                
                            quarter = -1 if dy > 0 else 1
                            gesture_status = f"Swipe {'Down' if dy > 0 else 'Up'} (Layer {layer})"

                        start_layer_animation(axis, layer, quarter)
                        record_move(axis, layer, quarter)
                        drag_move_triggered = True
                        last_drag_time = cur_time
                else:
                    is_pinching = False
                    drag_move_triggered = False
                    
                    cv2.circle(frame, (ix, iy), 6, (0, 255, 255), -1)
                    
                    if fingers >= 4:
                        YAW += 2.0
                        gesture_status = "Open Hand (Rotate View Right)"
                    elif fingers == 0:
                        YAW -= 2.0
                        gesture_status = "Fist (Rotate View Left)"
        else:
            is_pinching = False
            drag_move_triggered = False
            hover_button_id = None
            gesture_status = "No Hand Detected"

        if scanning_mode and game_mode == "PRACTICE":
            curr_face_name = face_names[scan_step_index] if scan_step_index < len(face_names) else "Done"
            cv2.putText(frame, f"ROTATE & SCAN FACE: {curr_face_name} (Pinch to Snap)", (40, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, COLOR_ACCENT, 2, cv2.LINE_AA)
            
            box_size = 180
            start_cx = CUBE_VIEW_WIDTH // 2 - box_size // 2
            start_cy = HEIGHT // 2 - box_size // 2
            cv2.rectangle(frame, (start_cx, start_cy), (start_cx + box_size, start_cy + box_size), (0, 255, 255), 3)

        render_scene()
        glfw.swap_buffers(window)

        glReadBuffer(GL_BACK)
        gl_pixels = glReadPixels(0, 0, CUBE_VIEW_WIDTH, HEIGHT, GL_BGR, GL_UNSIGNED_BYTE)
        gl_image = np.frombuffer(gl_pixels, dtype=np.uint8).reshape(HEIGHT, CUBE_VIEW_WIDTH, 3)
        gl_image = cv2.flip(gl_image, 0)

        alpha = cv2.cvtColor(gl_image, cv2.COLOR_BGR2GRAY)
        _, alpha = cv2.threshold(alpha, 1, 255, cv2.THRESH_BINARY)
        if alpha.shape[:2] != (HEIGHT, CUBE_VIEW_WIDTH):
            alpha = cv2.resize(alpha, (CUBE_VIEW_WIDTH, HEIGHT))

        alpha_inv = cv2.bitwise_not(alpha)
        left_bg = frame[:, :CUBE_VIEW_WIDTH]
        left_fg = gl_image
        
        background = cv2.bitwise_and(left_bg, left_bg, mask=alpha_inv)
        foreground = cv2.bitwise_and(left_fg, left_fg, mask=alpha)
        frame[:, :CUBE_VIEW_WIDTH] = cv2.add(background, foreground)

        draw_dashboard_overlay(frame, current_hover_id, hover_progress)

        cv2.rectangle(frame, (25, 25), (420, 85), COLOR_PANEL, -1)
        cv2.rectangle(frame, (25, 25), (420, 85), COLOR_ACCENT, 2)
        cv2.putText(frame, f"Status: {gesture_status}", (35, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, COLOR_TEXT, 2, cv2.LINE_AA)

        if time.time() < alert_expire_time:
            text_size, _ = cv2.getTextSize(active_alert_message, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            alert_w = text_size[0] + 50
            center_x = CUBE_VIEW_WIDTH // 2
            box_left = center_x - alert_w // 2
            box_right = center_x + alert_w // 2
            
            cv2.rectangle(frame, (box_left, 20), (box_right, 75), (15, 20, 35), -1)
            cv2.rectangle(frame, (box_left, 20), (box_right, 75), COLOR_RED, 2)
            cv2.putText(frame, active_alert_message, (box_left + 25, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow(window_name, frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord('q'):
            break

        glfw.poll_events()

    cap.release()
    cv2.destroyAllWindows()
    glfw.terminate()

In [96]:
if __name__ == "__main__":
    main()